# Model ladder comparison

One table, one held-out gold set (`data/gold/gold.tsv`, hand-written), every approach. Regenerate the inputs with `python -m absa_service.train --transformer --transformer-zero-shot` and `python -m evaluation.lexicon_failure`.

**Read with care:** n=94 gold sentences, one training seed, so differences of a few points are within noise; training data are template-generated weak labels.

In [ ]:
import json, pandas as pd
from pathlib import Path
RES = Path('../artifacts/results') if Path('../artifacts/results/model_ladder.json').exists() else Path('../results')
ladder = json.loads((RES/'model_ladder.json').read_text())
rows = []
for name, r in ladder.items():
    if name.startswith('_'): continue
    g = r['gold']
    rows.append({'model': name, 'accuracy': g['accuracy'], 'macro_f1': g['macro_f1'],
                 'trap_accuracy': g.get('trap_accuracy'), 'ms_per_item': g['latency_ms_per_item'],
                 **{f'{c}_f1': g['per_class'][c]['f1'] for c in ('negative','neutral','positive')}})
df = pd.DataFrame(rows).set_index('model').round(3)
df

In [ ]:
import matplotlib.pyplot as plt
ax = df[['macro_f1','trap_accuracy']].plot.bar(figsize=(9,4), ylim=(0,1))
ax.set_ylabel('score'); ax.set_title('Macro-F1 and accuracy on lexicon-trap sentences'); plt.tight_layout(); plt.show()

## Where do the lexicons fail?
Sentences flagged `trap=1` in the gold set are ones where financial semantics invert general-purpose polarity (falling costs/debt are good; rising litigation expense is bad).

In [ ]:
fail = json.loads((RES/'lexicon_failures.json').read_text())
pd.DataFrame({k: {'error_all': v['error_rate'], 'error_trap': v['error_rate_trap'], 'error_nontrap': v['error_rate_nontrap']} for k, v in fail['models'].items()}).T.round(3)

In [ ]:
pd.DataFrame(fail['models']['vader']['failures']).head(15)[['aspect','gold','predicted','sentence']]

## Training curves (weak-label validation accuracy)

In [ ]:
for name in ('cnn','lstm','finbert_ft'):
    h = ladder.get(name, {}).get('history') or []
    if h: print(name, [round(x.get('val_acc', float('nan')), 3) for x in h])